<a href="https://colab.research.google.com/github/angeloskalaf442-lang/HydroPyro/blob/main/HydroPyro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#!/usr/bin/env python3
import os
import sys
import subprocess
import requests
import numpy as np
import pandas as pd
from io import BytesIO
from datetime import datetime, timedelta
from PIL import Image

# === AUTO-SETUP PACKAGES ===
def ensure_packages():
    required = ["numpy", "pandas", "requests", "pillow", "tensorflow"]
    for pkg in required:
        try:
            __import__(pkg if pkg != "pillow" else "PIL")
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

ensure_packages()

import tensorflow as tf
from tensorflow.keras import layers, models

# === GLOBAL CONFIG ===
IMG_SIZE = (224, 224)
LSTM_SEQ_LEN = 30
LSTM_FEATURES = 8
MODEL_OUT = "trained_fire_flood_binary.keras"
HEADERS = {"User-Agent": "EnvironmentalAI-Bot/3.8"}

WEATHER_COLS = [
    "temperature_2m", "relative_humidity_2m", "dewpoint_2m", "precipitation",
    "windspeed_10m", "et0_fao_evapotranspiration", "surface_pressure", "soil_moisture_0_to_7cm"
]

WEATHER_MIN_MAX = {
    "temperature_2m": (-10, 50), "relative_humidity_2m": (0, 100),
    "dewpoint_2m": (-10, 30), "precipitation": (0, 50),
    "windspeed_10m": (0, 100), "et0_fao_evapotranspiration": (0, 15),
    "surface_pressure": (950, 1050), "soil_moisture_0_to_7cm": (0, 1)
}

# Labels: 0 = Fire, 1 = Flood
HISTORICAL_SAMPLES = [
    {"name": "Evia Fire 2021", "lat": 38.9, "lon": 23.3, "date": "2021-08-06", "label": 0},
    {"name": "Thessaly Flood 2023", "lat": 39.4, "lon": 22.1, "date": "2023-09-07", "label": 1},
    {"name": "Rhodes Fire 2023", "lat": 36.1, "lon": 27.9, "date": "2023-07-23", "label": 0},
    {"name": "Mandra Flood 2017", "lat": 38.1, "lon": 23.5, "date": "2017-11-15", "label": 1},
    {"name": "Alexandroupoli Fire 2023", "lat": 40.9, "lon": 25.9, "date": "2023-08-22", "label": 0},
    {"name": "Daniel Flood Volos 2023", "lat": 39.3, "lon": 22.9, "date": "2023-09-05", "label": 1},
]

# === CORE FUNCTIONS ===

def get_coords(query: str):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={query}&count=1&language=el&format=json"
    r = requests.get(url, headers=HEADERS, timeout=10)
    data = r.json().get("results")
    if not data: raise Exception("Η περιοχή δεν βρέθηκε.")
    return float(data[0]["latitude"]), float(data[0]["longitude"]), data[0]["name"]

def fetch_weather_data(lat, lon, date_obj):
    now = datetime.now()
    is_future = date_obj.date() > now.date()
    url = "https://api.open-meteo.com/v1/forecast" if is_future else "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat, "longitude": lon,
        "hourly": ",".join(WEATHER_COLS), "timezone": "UTC"
    }

    if is_future:
        params["forecast_days"] = 3
    else:
        params["start_date"] = (date_obj - timedelta(days=2)).strftime("%Y-%m-%d")
        params["end_date"] = date_obj.strftime("%Y-%m-%d")

    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=15)
        res = r.json().get("hourly", {})
        if not res: return np.zeros((LSTM_SEQ_LEN, LSTM_FEATURES), dtype=np.float32)

        df = pd.DataFrame(res)[WEATHER_COLS].ffill().bfill()
        for col in WEATHER_COLS:
            mi, ma = WEATHER_MIN_MAX[col]
            df[col] = (df[col] - mi) / (ma - mi + 1e-7)

        data = np.nan_to_num(df.values.astype(np.float32))
        return data[-LSTM_SEQ_LEN:]
    except:
        return np.zeros((LSTM_SEQ_LEN, LSTM_FEATURES), dtype=np.float32)

def fetch_nasa_image(lat, lon, date_obj):
    q_date = min(date_obj, datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    url = "https://wvs.earthdata.nasa.gov/api/v1/snapshot"
    box = 0.2
    params = {
        "REQUEST": "GetSnapshot", "TIME": q_date,
        "BBOX": f"{lon-box},{lat-box},{lon+box},{lat+box}",
        "CRS": "EPSG:4326", "LAYERS": "MODIS_Terra_CorrectedReflectance_TrueColor",
        "FORMAT": "image/jpeg", "WIDTH": 224, "HEIGHT": 224
    }
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=15)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        return np.array(img.resize(IMG_SIZE)) / 255.0
    except:
        return np.zeros((224, 224, 3), dtype=np.float32)

def build_hybrid_model():
    # CNN for Terrain Analysis
    img_in = layers.Input(shape=(224, 224, 3), name="image_input")
    x = layers.Conv2D(32, (3,3), activation='relu', kernel_initializer='he_normal')(img_in)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # LSTM for Weather Trends
    seq_in = layers.Input(shape=(LSTM_SEQ_LEN, LSTM_FEATURES), name="sensor_input")
    y = layers.LSTM(64, kernel_initializer='glorot_uniform')(seq_in)

    # Binary Classification Merging
    combined = layers.Concatenate()([x, y])
    z = layers.Dense(64, activation='relu')(combined)
    out = layers.Dense(2, activation='softmax', name="output")(z) # Binary: Fire/Flood

    model = models.Model(inputs=[img_in, seq_in], outputs=out)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model



def train_model():
    print("\n🚀 Εκκίνηση εκπαίδευσης (Binary Fire/Flood)...")
    model = build_hybrid_model()

    x_img, x_weather, y_labels = [], [], []
    for s in HISTORICAL_SAMPLES:
        print(f"  -> Data Fetch: {s['name']}...")
        d = datetime.strptime(s['date'], "%Y-%m-%d")
        x_img.append(fetch_nasa_image(s['lat'], s['lon'], d))
        x_weather.append(fetch_weather_data(s['lat'], s['lon'], d))
        y_labels.append(s['label'])

    model.fit(
        {"image_input": np.array(x_img), "sensor_input": np.array(x_weather)},
        np.array(y_labels),
        epochs=15, verbose=1
    )
    model.save(MODEL_OUT)
    return model

# === MAIN ===
def main():
    print("\n--- 🛡️ AI DISASTER CLASSIFIER (FIRE vs FLOOD) ---")

    if not os.path.exists(MODEL_OUT):
        model = train_model()
    else:
        model = tf.keras.models.load_model(MODEL_OUT)

    place = input("\n📍 Περιοχή: ").strip()
    try:
        lat, lon, name = get_coords(place)
        date_str = input("📅 Ημερομηνία Πρόβλεψης (YYYY-MM-DD) ή Enter: ").strip()
        t_date = datetime.strptime(date_str, "%Y-%m-%d") if date_str else datetime.now()

        print("📡 Ανάλυση...")
        img = fetch_nasa_image(lat, lon, t_date)
        weather = fetch_weather_data(lat, lon, t_date)

        res = model.predict({"image_input": img[None,...], "sensor_input": weather[None,...]}, verbose=0)[0]

        print(f"\n📊 ΠΟΡΙΣΜΑ {name.upper()}:")
        print(f"🔥 Πιθανότητα Φωτιάς: {res[0]:.1%}")
        print(f"🌊 Πιθανότητα Πλημμύρας: {res[1]:.1%}")

    except Exception as e:
        print(f"❌ Σφάλμα: {e}")

if __name__ == "__main__":
    main()


--- 🛡️ AI DISASTER CLASSIFIER (FIRE vs FLOOD) ---

🚀 Εκκίνηση εκπαίδευσης (Binary Fire/Flood)...
  -> Data Fetch: Evia Fire 2021...
  -> Data Fetch: Thessaly Flood 2023...
  -> Data Fetch: Rhodes Fire 2023...
  -> Data Fetch: Mandra Flood 2017...
  -> Data Fetch: Alexandroupoli Fire 2023...
  -> Data Fetch: Daniel Flood Volos 2023...
Epoch 1/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.5000 - loss: 0.7001
Epoch 2/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6489   
Epoch 3/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 703ms/step - accuracy: 0.6667 - loss: 0.6218
Epoch 4/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6667 - loss: 0.5931   
Epoch 5/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 1.0000 - loss: 0.5658
Epoch 6/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 1.0000 - loss: 0.5358
Epoch 7/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 1.0000 - loss: 0.5011
Epoch 8/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 1.0000 - loss: 0